In [ ]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append("..")

In [ ]:
import os
import sys
import json
import logging

import torch
import transformers

from rich.console import Console
from rich.theme import Theme
from rich.markup import escape

from src.utils import logging_utils

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["CUDA_VISIBLE_DEVICES"] = "1,2,3,4,5,6"

custom_theme = Theme({"default": "white"})
console = Console(theme=custom_theme, highlight=False)

def rich_print(*args, **kwargs):
    console.print(*args, **kwargs)

logging.basicConfig(
    level=logging.INFO,
    format=logging_utils.DEFAULT_FORMAT,
    datefmt=logging_utils.DEFAULT_DATEFMT,
    stream=sys.stdout,
)
logger = logging.getLogger(__name__)
logger.info(f"{torch.__version__=}, {torch.version.cuda=}")
logger.info(f"{transformers.__version__=}")

## Model

Load the LM under study. Uncomment to load — large model, takes time + VRAM.

In [ ]:
from src.models import ModelandTokenizer

MODEL_KEY = "Qwen/Qwen2.5-Coder-32B-Instruct"

mt = ModelandTokenizer(
    model_key=MODEL_KEY,
    dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="eager",
)

## Dataset

`design_patterns.json` holds two concepts:
- `monolithic_vs_modular` — one big function vs split helpers/classes
- `design_pattern` — observer / singleton / strategy / factory / decorator / visitor / iterator / if_else / callback_list / subclassing

Samples use the new `prompt_stub` field (typically `"class "` or `""`) instead of per-language `stubs`, since these tasks aren't tied to a single function signature and we don't compile them.

In [ ]:
from src.goals.concept_data import GoalConceptDataset, GoalConceptSample

DATA_PATH = "../data/goal_concepts/design_patterns.json"
ds = GoalConceptDataset.load(DATA_PATH)

from collections import Counter
print(f"{len(ds)} samples")
print("by concept:", Counter(s.concept for s in ds))
for s in ds:
    print(f"  [{s.concept:22s}] stub={s.prompt_stub!r:10s} {s.instruction[:70]}...")

## Prompt construction

`build_prompt` mirrors `14_iter_rec_concept` but goes through `get_code_stub`, which now falls back to `prompt_stub` when no per-language stub is registered. Set `preferred_strategy=None` for the **proactive** (unbiased) prompt; set it to a concept-valid key (e.g. `"observer"`) for the **directed** prompt.

In [ ]:
from src.goals.concept_data import StrategyDirective
from src.goals.functional import apply_chat_template
from src.tokens import prepare_input

_SYSTEM_PROMPT_PY = (
    "You are a helpful assistant proficient in Python. When completing code, "
    "ensure proper syntax and logical correctness. Make sure to have all codes "
    "inside the <code> </code> tags."
)

def build_prompt(
    mt,
    sample: GoalConceptSample,
    lang: str = "python",
    add_docstring: bool = False,
    without_comments: bool = False,
    preferred_strategy: StrategyDirective | None = None,
    system_prompt: str = _SYSTEM_PROMPT_PY,
) -> str:
    """Build a chat prompt for *sample*. Supports both per-language stubs and prompt_stub."""
    stub = sample.get_code_stub(lang, add_docstring=add_docstring)
    wc_cache = sample.without_comments
    ps_cache = sample.preferred_strategy
    sample.without_comments = without_comments
    sample.preferred_strategy = preferred_strategy
    try:
        prompt = apply_chat_template(
            mt,
            user_query=sample.full_instruction,
            incomplete_code=stub,
            system_prompt=system_prompt,
        )
    finally:
        sample.without_comments = wc_cache
        sample.preferred_strategy = ps_cache
    return prompt


@torch.inference_mode()
def sample_completions(
    mt,
    prompt: str,
    n: int = 8,
    temperature: float = 0.7,
    top_p: float = 0.9,
    max_new_tokens: int = 400,
    stop_tag: str = "</code>",
) -> list[str]:
    """Generate n independent completions via sampling. Returns text continuations only."""
    inputs = prepare_input(prompts=prompt, tokenizer=mt)
    input_ids = inputs["input_ids"].to(mt._model.device).expand(n, -1)
    attention_mask = inputs["attention_mask"].to(mt._model.device).expand(n, -1)
    prompt_len = input_ids.shape[1]
    outputs = mt._model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=temperature,
        top_p=top_p,
        pad_token_id=mt.tokenizer.eos_token_id,
    )
    results = []
    for out in outputs:
        text = mt.tokenizer.decode(out[prompt_len:], skip_special_tokens=True)
        if stop_tag in text:
            text = text[: text.index(stop_tag)]
        results.append(text.rstrip())
    return results

## Smoke test: one sample, both prompt modes

Compare the proactive (no directive) and directed (e.g. `observer`) prompts side-by-side.

In [ ]:
sample = ds[4]   # NotificationCenter — observer-vs-callback_list pair

p_proactive = build_prompt(mt, sample)
p_directed  = build_prompt(mt, sample, preferred_strategy="observer")

print("=== PROACTIVE ===")
print(p_proactive)
print("\n=== DIRECTED (observer) ===")
print(p_directed)

## Code extraction + pattern labeling

Heuristic labelers — kept inline so you can iterate on the indicators. They label each completion as a small dict of bool flags; aggregate over many samples to estimate `rate_pattern_X` per task.

**Caveats:**
- These are *surface* indicators (class names, method names, decorators). A direction trained on a labeler that fires on the token `notify(` may capture lexical surface rather than structural intent — exactly the competence-vs-intent confound from the existing pipeline. Use these labels for *task selection* first (which tasks reliably split), not as ground-truth probe labels.
- AST parse fails (incomplete generation, leftover `<code>` artifacts) → labeler falls back to regex-only.

In [ ]:
import ast
import re
from dataclasses import dataclass, field, asdict
from typing import Optional


_CODE_TAG_RE = re.compile(r"<code>\s*(.*?)\s*(?:</code>|$)", re.DOTALL)

def extract_code(generation: str, prompt_stub: str = "") -> str:
    """Pull text inside <code>...</code>; fall back to raw generation. Prepend the
    stub so the AST parser sees a syntactically plausible chunk."""
    m = _CODE_TAG_RE.search(generation)
    body = m.group(1) if m else generation
    return (prompt_stub + body).strip()


@dataclass
class CodeStructure:
    """Cheap structural summary used by the per-concept labelers."""
    n_classes: int = 0
    n_top_level_defs: int = 0
    n_methods: int = 0          # def inside any class
    n_nested_defs: int = 0      # def inside another def
    class_names: list[str] = field(default_factory=list)
    method_names: list[str] = field(default_factory=list)
    function_names: list[str] = field(default_factory=list)
    decorator_uses: list[str] = field(default_factory=list)
    ast_ok: bool = True


def summarize_structure(code: str) -> CodeStructure:
    try:
        tree = ast.parse(code)
    except SyntaxError:
        # regex fallback (very approximate)
        return CodeStructure(
            n_classes=len(re.findall(r"^class\s+(\w+)", code, re.M)),
            n_top_level_defs=len(re.findall(r"^def\s+(\w+)", code, re.M)),
            class_names=re.findall(r"^class\s+(\w+)", code, re.M),
            function_names=re.findall(r"^def\s+(\w+)", code, re.M),
            decorator_uses=re.findall(r"^\s*@(\w+)", code, re.M),
            ast_ok=False,
        )

    s = CodeStructure()
    for node in ast.iter_child_nodes(tree):
        if isinstance(node, ast.ClassDef):
            s.n_classes += 1
            s.class_names.append(node.name)
            for ch in ast.walk(node):
                if isinstance(ch, ast.FunctionDef):
                    s.n_methods += 1
                    s.method_names.append(ch.name)
        elif isinstance(node, ast.FunctionDef):
            s.n_top_level_defs += 1
            s.function_names.append(node.name)
            for ch in ast.walk(node):
                if isinstance(ch, ast.FunctionDef) and ch is not node:
                    s.n_nested_defs += 1

    for node in ast.walk(tree):
        for d in getattr(node, "decorator_list", []) or []:
            if isinstance(d, ast.Name):
                s.decorator_uses.append(d.id)
            elif isinstance(d, ast.Attribute):
                s.decorator_uses.append(d.attr)
    return s


# ── Per-concept labelers ─────────────────────────────────────────────────────

def label_monolithic_vs_modular(code: str) -> dict:
    s = summarize_structure(code)
    n_units = s.n_classes + s.n_top_level_defs
    is_monolithic = (n_units <= 1) and (s.n_methods + s.n_nested_defs <= 1)
    is_modular    = (s.n_classes >= 1) or (s.n_top_level_defs >= 2) or (s.n_nested_defs >= 1)
    return {
        "is_monolithic":   is_monolithic,
        "is_modular":      is_modular,
        "n_classes":       s.n_classes,
        "n_top_defs":      s.n_top_level_defs,
        "n_methods":       s.n_methods,
        "ast_ok":          s.ast_ok,
    }


_OBSERVER_HINTS  = (r"subscribe\b", r"unsubscribe\b", r"notify\b", r"\bObserver\b", r"\bSubject\b")
_SINGLETON_HINTS = (r"_instance\s*=\s*None", r"__new__\b", r"getInstance\b", r"\bSingleton\b")
_FACTORY_HINTS   = (r"\bFactory\b", r"\.create\(", r"\.make\(", r"create_\w+\(")
_VISITOR_HINTS   = (r"\bvisit_\w+", r"\bVisitor\b", r"\.accept\(")
_DECORATOR_HINTS = (r"^\s*@\w+", r"functools\.wraps", r"def wrapper\(")
_STRATEGY_HINTS  = (r"strategies\s*=\s*{", r"handlers\s*=\s*{", r"\bStrategy\b")
_ITERATOR_HINTS  = (r"__iter__\b", r"__next__\b", r"\byield\b")

_IFELSE_RE = re.compile(r"\belif\b", re.M)

def _any(patterns: tuple[str, ...], code: str) -> bool:
    return any(re.search(p, code, re.M) for p in patterns)


def label_design_pattern(code: str) -> dict:
    s = summarize_structure(code)
    flags = {
        "has_observer":      _any(_OBSERVER_HINTS, code),
        "has_singleton":     _any(_SINGLETON_HINTS, code),
        "has_factory":       _any(_FACTORY_HINTS, code),
        "has_visitor":       _any(_VISITOR_HINTS, code),
        "has_decorator":     _any(_DECORATOR_HINTS, code) or len(s.decorator_uses) > 0,
        "has_strategy_map":  _any(_STRATEGY_HINTS, code),
        "has_iterator":      _any(_ITERATOR_HINTS, code),
        "has_if_else_chain": len(_IFELSE_RE.findall(code)) >= 2,
        "has_callback_list": bool(re.search(r"(callbacks|subscribers|listeners)\s*=\s*(\[|\{)", code)),
        "n_classes":         s.n_classes,
        "ast_ok":            s.ast_ok,
    }
    return flags


_LABELERS: dict[str, callable] = {
    "monolithic_vs_modular": label_monolithic_vs_modular,
    "design_pattern":        label_design_pattern,
}

def label_completion(concept: str, code: str) -> dict:
    fn = _LABELERS.get(concept)
    if fn is None:
        raise KeyError(f"No labeler for concept={concept!r}. Registered: {list(_LABELERS)}")
    return fn(code)

## Run: sample completions for every task and label

For each task: generate N proactive completions, label each, store per-flag rates. Result is a DataFrame ready to filter on ("which tasks split cleanly along observer-vs-callback?").

In [ ]:
import pandas as pd
from tqdm.auto import tqdm

N_SAMPLES = 8

raw_records = []   # one row per (sample, completion)
for sample in tqdm(ds, desc="tasks"):
    prompt = build_prompt(mt, sample)
    completions = sample_completions(mt, prompt, n=N_SAMPLES)
    for i, gen in enumerate(completions):
        code = extract_code(gen, prompt_stub=sample.prompt_stub or "")
        labels = label_completion(sample.concept, code)
        raw_records.append({
            "concept":      sample.concept,
            "task":         sample.instruction[:50] + "...",
            "completion":   i,
            "code":         code,
            **labels,
        })

raw_df = pd.DataFrame(raw_records)
raw_df.head()

In [ ]:
# Aggregate: per-task rate of every boolean flag.
BOOL_COLS = [c for c in raw_df.columns if raw_df[c].dtype == bool]

rate_df = (
    raw_df.groupby(["concept", "task"])[BOOL_COLS]
    .mean()
    .round(2)
    .reset_index()
)
rate_df

## Inspect a single completion

Quick eyeball check — pick a row from `raw_df` and print the extracted code + its labels.

In [ ]:
row = raw_df.iloc[0]
rich_print(f"[bold]task   :[/bold] {row['task']}")
rich_print(f"[bold]concept:[/bold] {row['concept']}")
rich_print(f"[bold]labels :[/bold] {{ {', '.join(f'{c}={row[c]}' for c in BOOL_COLS if row[c])} }}")
print("-" * 80)
print(row["code"])

## Next steps

1. **Task selection.** Keep tasks whose `rate_df` row shows a non-degenerate split (e.g. `has_observer ∈ [0.2, 0.8]`). These are the candidates where the model has a *real* choice — and therefore a stable internal commitment worth probing.
2. **Pivot detection.** For each retained task, find the earliest token position where the conditional `p(continuation | pattern A) >> p(continuation | pattern B)` — likely the first generated identifier (class name, def name, decorator).
3. **Cross-task transfer.** Mine a steering vector from one task's pivot in pattern A; patch it onto another task's pivot. If generation flips toward pattern A, that's evidence of a structural pattern direction (not just task-specific lexical bias).
4. **Lexical confound check.** Test transfer to tasks where you've masked / renamed the obvious surface tokens (`Observer` → `Subject`, `notify` → `broadcast`). If the direction still fires, it tracks intent over surface.